# 03 · Cadenas: recorridos, palíndromos y búsqueda de patrones

Tópico 03 del temario.

## Lo esencial de los strings

Un string es una **secuencia inmutable** de caracteres. Se indexa como una lista, pero no se puede modificar en el sitio.

In [ ]:
s = "competencia"

assert len(s) == 11
assert s[0] == "c" and s[-1] == "a"
assert s[2:5] == "mpe" and s[:3] == "com" and s[-3:] == "cia"
assert s[::-1] == "aicnetepmoc"

# Recorrer
assert [c for c in "abc"] == ["a", "b", "c"]
assert [(i, c) for i, c in enumerate("ab")] == [(0, "a"), (1, "b")]

### Métodos útiles

| Método | Qué hace |
|---|---|
| `s.lower()`, `s.upper()` | pasa a minúsculas / mayúsculas |
| `s.strip()` | quita espacios (y saltos de línea) de los extremos |
| `s.split()`, `s.split(",")` | parte en una lista |
| `",".join(lista)` | une una lista de strings |
| `s.count(t)` | cuántas veces aparece `t` (**sin solapar**) |
| `s.find(t)` | primera posición de `t`, o `-1` |
| `s.replace(a, b)` | reemplaza |
| `s.startswith(t)`, `s.endswith(t)` | prefijo / sufijo |
| `s.isdigit()`, `s.isalpha()` | clasificar |

In [ ]:
assert "Hola Mundo".lower() == "hola mundo"
assert "a,b,c".split(",") == ["a", "b", "c"]
assert "-".join(["x", "y"]) == "x-y"
assert "banana".count("an") == 2 and "banana".find("na") == 2 and "banana".find("z") == -1
assert "banana".replace("a", "o") == "bonono"
assert "abc".startswith("ab") and "abc".endswith("bc")

### Caracteres y números: `ord` y `chr`

In [ ]:
assert ord("a") == 97 and chr(97) == "a"
assert ord("c") - ord("a") == 2                # posición de la letra en el alfabeto (0-based)
assert chr(ord("a") + 3) == "d"

# Contar frecuencias con un arreglo de 26 posiciones
def frecuencias(s):
    cuenta = [0] * 26
    for c in s:
        cuenta[ord(c) - ord("a")] += 1
    return cuenta


f = frecuencias("abracadabra")
assert f[0] == 5 and f[1] == 2 and f[17] == 2          # a, b, r

# o con un diccionario / Counter
from collections import Counter
assert Counter("abracadabra")["a"] == 5

### Construir strings eficientemente

`s += c` dentro de un bucle largo es `O(n²)`. Usa una lista y `join`.

In [ ]:
partes = []
for i in range(5):
    partes.append(str(i))
assert "".join(partes) == "01234"

## Palíndromos

Un **palíndromo** se lee igual al derecho y al revés.

### Con slicing

In [ ]:
def es_palindromo(s):
    return s == s[::-1]


assert es_palindromo("reconocer") and not es_palindromo("python")
assert es_palindromo("") and es_palindromo("a")

### Con dos punteros (sin crear copia)

Compara el primero con el último, avanza hacia el centro.

In [ ]:
def es_palindromo_2p(s):
    i, j = 0, len(s) - 1
    while i < j:
        if s[i] != s[j]:
            return False
        i += 1
        j -= 1
    return True


assert es_palindromo_2p("reconocer") and not es_palindromo_2p("python")

### Ignorando espacios y mayúsculas

In [ ]:
def es_palindromo_frase(frase):
    limpia = [c.lower() for c in frase if c.isalnum()]
    return limpia == limpia[::-1]


assert es_palindromo_frase("Anita lava la tina")
assert es_palindromo_frase("A man, a plan, a canal: Panama")

### Subcadena palíndroma más larga: expandir desde el centro

Cada palíndromo tiene un centro (una letra, o entre dos letras). Expande mientras coincidan. `O(n²)`.

In [ ]:
def palindromo_mas_largo(s):
    mejor = ""
    for centro in range(len(s)):
        for izq, der in ((centro, centro), (centro, centro + 1)):      # impar y par
            while izq >= 0 and der < len(s) and s[izq] == s[der]:
                izq -= 1
                der += 1
            if der - izq - 1 > len(mejor):
                mejor = s[izq + 1:der]
    return mejor


assert palindromo_mas_largo("babad") in ("bab", "aba")
assert palindromo_mas_largo("cbbd") == "bb"
assert palindromo_mas_largo("a") == "a"

### Contar subcadenas palíndromas

In [ ]:
def contar_palindromos(s):
    total = 0
    for centro in range(len(s)):
        for izq, der in ((centro, centro), (centro, centro + 1)):
            while izq >= 0 and der < len(s) and s[izq] == s[der]:
                total += 1
                izq -= 1
                der += 1
    return total


assert contar_palindromos("aaa") == 6         # a a a aa aa aaa
assert contar_palindromos("abc") == 3

## Búsqueda de patrones

Problema: encontrar en qué posiciones del `texto` aparece el `patron`.

### Con los métodos de Python

In [ ]:
texto = "abababa"

assert texto.find("aba") == 0
assert texto.find("aba", 1) == 2                 # busca desde la posición 1
assert texto.count("aba") == 2                   # count NO cuenta solapadas (hay 3)

### Búsqueda ingenua: probar cada posición

Desliza el patrón por el texto y compara. `O(n · m)`, suficiente para textos de unos pocos miles.

In [ ]:
def posiciones(texto, patron):
    n, m = len(texto), len(patron)
    res = []
    for i in range(n - m + 1):
        if texto[i:i + m] == patron:
            res.append(i)
    return res


def contar_ocurrencias(texto, patron):
    return len(posiciones(texto, patron))


assert posiciones("abababa", "aba") == [0, 2, 4]
assert contar_ocurrencias("aaaa", "aa") == 3        # con solapamiento
assert posiciones("abc", "z") == []

### Para curiosos: la función prefijo (KMP)

La búsqueda ingenua repite comparaciones. La **función prefijo** `pi[i]` guarda la longitud del prefijo más largo que también es sufijo de `s[:i+1]`. Con ella, buscar es `O(n + m)`.

In [ ]:
def funcion_prefijo(s):
    pi = [0] * len(s)
    k = 0
    for i in range(1, len(s)):
        while k and s[i] != s[k]:
            k = pi[k - 1]
        if s[i] == s[k]:
            k += 1
        pi[i] = k
    return pi


def posiciones_kmp(texto, patron):
    union = patron + "#" + texto
    pi = funcion_prefijo(union)
    m = len(patron)
    return [i - 2 * m for i in range(len(union)) if pi[i] == m]


assert funcion_prefijo("abab") == [0, 0, 1, 2]
assert posiciones_kmp("abababa", "aba") == [0, 2, 4]

## Más técnicas

### Anagramas

Dos palabras son **anagramas** si tienen las mismas letras en distinto orden.

In [ ]:
def son_anagramas(a, b):
    return sorted(a) == sorted(b)            # O(n log n)


def son_anagramas_rapido(a, b):
    return Counter(a) == Counter(b)          # O(n)


assert son_anagramas("roma", "amor") and not son_anagramas("roma", "rama2")
assert son_anagramas_rapido("roma", "amor")

### Compresión por repeticiones (run-length)

`"aaabbc"` → `"a3b2c1"`

In [ ]:
def comprimir(s):
    if not s:
        return ""
    res = []
    actual, cuenta = s[0], 1
    for c in s[1:]:
        if c == actual:
            cuenta += 1
        else:
            res.append(actual + str(cuenta))
            actual, cuenta = c, 1
    res.append(actual + str(cuenta))
    return "".join(res)


assert comprimir("aaabbc") == "a3b2c1"
assert comprimir("") == "" and comprimir("z") == "z1"

### Cifrado César

Desplaza cada letra `k` posiciones en el alfabeto (módulo 26).

In [ ]:
def cesar(s, k):
    res = []
    for c in s:
        if "a" <= c <= "z":
            res.append(chr((ord(c) - ord("a") + k) % 26 + ord("a")))
        else:
            res.append(c)
    return "".join(res)


assert cesar("abc", 1) == "bcd"
assert cesar("xyz", 3) == "abc"
assert cesar(cesar("hola mundo", 5), -5) == "hola mundo"

## Problemas tipo (estilo grader)

**P1.** `primera_unica(s)`: índice del primer carácter que aparece una sola vez, o `-1`.
**P2.** `es_rotacion(a, b)`: `True` si `b` es `a` rotada (por ejemplo `"abcd"` y `"cdab"`).
**P3.** `prefijo_comun(palabras)`: el prefijo común más largo de una lista de palabras.

In [ ]:
def primera_unica(s):
    cuenta = Counter(s)
    for i, c in enumerate(s):
        if cuenta[c] == 1:
            return i
    return -1


def es_rotacion(a, b):
    return len(a) == len(b) and b in a + a


def prefijo_comun(palabras):
    if not palabras:
        return ""
    pref = palabras[0]
    for p in palabras[1:]:
        while not p.startswith(pref):
            pref = pref[:-1]
    return pref


assert primera_unica("aabbcd") == 4 and primera_unica("aabb") == -1
assert es_rotacion("abcd", "cdab") and not es_rotacion("abcd", "acbd")
assert prefijo_comun(["flor", "flota", "flujo"]) == "fl"
assert prefijo_comun(["perro", "gato"]) == ""
print("ok")

## Para practicar

1. `invertir_palabras(frase)`: invierte el orden de las palabras.
2. `vocales_y_consonantes(s)`: cuántas de cada una.
3. `subcadena_sin_repetir(s)`: longitud de la subcadena más larga sin letras repetidas (ver cuaderno 04).
4. `mayuscula_cada_palabra(frase)` sin usar `title()`.
5. `contar_subcadenas_iguales(s, k)`: cuántas subcadenas de longitud `k` aparecen más de una vez.

## Errores típicos

- Usar `s.count(p)` creyendo que cuenta ocurrencias solapadas.
- Olvidar `.strip()` al leer líneas (el `\n` final cuenta como carácter).
- Intentar `s[0] = "x"`: los strings no se modifican.
- Armar un string grande con `+=` en un bucle largo.
- Confundir `s[i:j]` (excluye `j`) y salirse de rango con `i + m`.